# Waze user churn
## Data inspection and preparation
**Hernán Licastro | Educational portfolio**

Synthetic data and fictional course scenario. Not affiliated with Waze.

**Business question:** What data quality issues and preliminary usage patterns should the team understand before building a monthly churn model?

Scope: import, inspect, summarize and organize. No predictive model, hypothesis test or causal inference is delivered.

## Plan
Review the dictionary and proposal. One row represents one user. Separate monthly measures from lifetime measures and confirm prediction timing. Preserve raw data.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
paths = [Path("../data/waze_dataset.csv"), Path("data/waze_dataset.csv"), Path("waze_dataset.csv")]
path = next((p for p in paths if p.exists()), None)
if path is None:
    raise FileNotFoundError("No se encontró waze_dataset.csv")
df = pd.read_csv(path)

In [2]:
df.head(10)

,ID,label,sessions,drives,total_sessions,n_days_after_onboarding,total_navigations_fav1,total_navigations_fav2,driven_km_drives,duration_minutes_drives,activity_days,driving_days,device
0,0,retained,283,226,296.748273,2276,208,0,2628.845068,1985.775061,28,19,Android
1,1,retained,133,107,326.896596,1225,19,64,13715.920550,3160.472914,13,11,iPhone
2,2,retained,114,95,135.522926,2651,0,0,3059.148818,1610.735904,14,8,Android
3,3,retained,49,40,67.589221,15,322,7,913.591123,587.196542,7,3,iPhone
4,4,retained,84,68,168.247020,1562,166,5,3950.202008,1219.555924,27,18,Android
5,5,retained,113,103,279.544437,2637,0,0,901.238699,439.101397,15,11,iPhone
6,6,retained,3,2,236.725314,360,185,18,5249.172828,726.577205,28,23,iPhone
7,7,retained,39,35,176.072845,2999,0,0,7892.052468,2466.981741,22,20,iPhone
8,8,retained,57,46,183.532018,424,0,26,2651.709764,1594.342984,25,20,Android
9,9,churned,84,68,244.802115,2997,72,0,6043.460295,2341.838528,7,3,iPhone


In [3]:
df.info()
pd.DataFrame({"dtype": df.dtypes.astype(str), "non_null": df.notna().sum(), "missing": df.isna().sum(), "missing_pct": df.isna().mean()*100})

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14999 entries, 0 to 14998
Data columns (total 13 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   ID                       14999 non-null  int64  
 1   label                    14299 non-null  object 
 2   sessions                 14999 non-null  int64  
 3   drives                   14999 non-null  int64  
 4   total_sessions           14999 non-null  float64
 5   n_days_after_onboarding  14999 non-null  int64  
 6   total_navigations_fav1   14999 non-null  int64  
 7   total_navigations_fav2   14999 non-null  int64  
 8   driven_km_drives         14999 non-null  float64
 9   duration_minutes_drives  14999 non-null  float64
 10  activity_days            14999 non-null  int64  
 11  driving_days             14999 non-null  int64  
 12  device                   14999 non-null  object 
dtypes: float64(3), int64(8), object(2)
memory usage: 1.5+ MB


,dtype,non_null,missing,missing_pct
ID,int64,14999,0,0.000000
label,object,14299,700,4.666978
sessions,int64,14999,0,0.000000
drives,int64,14999,0,0.000000
total_sessions,float64,14999,0,0.000000
n_days_after_onboarding,int64,14999,0,0.000000
total_navigations_fav1,int64,14999,0,0.000000
total_navigations_fav2,int64,14999,0,0.000000
driven_km_drives,float64,14999,0,0.000000
duration_minutes_drives,float64,14999,0,0.000000


In [4]:
raw = df.iloc[:, :13]
pd.Series({
    "rows": len(raw), "columns": raw.shape[1],
    "duplicate_rows": int(raw.duplicated().sum()),
    "duplicate_ids": int(raw.ID.duplicated().sum()),
    "missing_labels": int(raw.label.isna().sum()),
    "zero_drives_positive_km": int(((raw.drives == 0) & (raw.driven_km_drives > 0)).sum()),
    "zero_driving_days_positive_km": int(((raw.driving_days == 0) & (raw.driven_km_drives > 0)).sum())
})

rows                             14999
columns                             13
duplicate_rows                       0
duplicate_ids                        0
missing_labels                     700
zero_drives_positive_km            106
zero_driving_days_positive_km     1024
dtype: int64

## Analyze
14,999 users and 13 original columns. Only label has missing values: 700 (4.67%). No duplicate rows or repeated IDs. ID is an identifier. total_sessions is a model estimate, so fractional values are expected. No rows are removed and no labels imputed.

In [5]:
df.describe()

,ID,sessions,drives,total_sessions,n_days_after_onboarding,total_navigations_fav1,total_navigations_fav2,driven_km_drives,duration_minutes_drives,activity_days,driving_days
count,14999.000000,14999.000000,14999.000000,14999.000000,14999.000000,14999.000000,14999.000000,14999.000000,14999.000000,14999.000000,14999.000000
mean,7499.000000,80.633776,67.281152,189.964447,1749.837789,121.605974,29.672512,4039.340921,1860.976012,15.537102,12.179879
std,4329.982679,80.699065,65.913872,136.405128,1008.513876,148.121544,45.394651,2502.149334,1446.702288,9.004655,7.824036
min,0.000000,0.000000,0.000000,0.220211,4.000000,0.000000,0.000000,60.441250,18.282082,0.000000,0.000000
25%,3749.500000,23.000000,20.000000,90.661156,878.000000,9.000000,0.000000,2212.600607,835.996260,8.000000,5.000000
50%,7499.000000,56.000000,48.000000,159.568115,1741.000000,71.000000,9.000000,3493.858085,1478.249859,16.000000,12.000000
75%,11248.500000,112.000000,93.000000,254.192341,2623.500000,178.000000,43.000000,5289.861262,2464.362632,23.000000,19.000000
max,14998.000000,743.000000,596.000000,1216.154633,3500.000000,1236.000000,415.000000,21183.401890,15851.727160,31.000000,30.000000


In [6]:
df.groupby(df.label.isna()).mean(numeric_only=True).drop(columns="ID")

,sessions,drives,total_sessions,n_days_after_onboarding,total_navigations_fav1,total_navigations_fav2,driven_km_drives,duration_minutes_drives,activity_days,driving_days
label,,,,,,,,,,
False,80.623820,67.255822,189.547409,1751.822505,121.747395,29.638296,4044.401535,1864.199794,15.544653,12.182530
True,80.837143,67.798571,198.483348,1709.295714,118.717143,30.371429,3935.967029,1795.123358,15.382857,12.125714


In [7]:
pd.crosstab(df.label.isna().rename("missing_label"), df.device, normalize="index")*100

device,Android,iPhone
missing_label,,
False,35.484999,64.515001
True,36.142857,63.857143


Summary statistics and device composition are broadly similar across missing-label groups. This does not establish that labels are missing completely at random.

In [8]:
counts = df.label.value_counts()
pd.DataFrame({"users": counts, "pct_known_labels": counts/counts.sum()*100})

,users,pct_known_labels
label,,
retained,11763,82.264494
churned,2536,17.735506


In [9]:
df.groupby("label").median(numeric_only=True).drop(columns="ID")

,sessions,drives,total_sessions,n_days_after_onboarding,total_navigations_fav1,total_navigations_fav2,driven_km_drives,duration_minutes_drives,activity_days,driving_days
label,,,,,,,,,,
churned,59.0,50.0,164.339042,1321.0,84.5,11.0,3652.655666,1607.183785,8.0,6.0
retained,56.0,47.0,157.586756,1843.0,68.0,9.0,3464.684614,1458.046141,17.0,14.0


Among 14,299 known labels, 17.74% churned and 82.26% were retained. Median activity days are 8 versus 17, driving days 6 versus 14, and tenure 1,321 versus 1,843 days. Monthly drives are slightly higher for churned users, 50 versus 47. Associations do not establish causes.

In [10]:
df["km_per_drive"] = df["driven_km_drives"] / df["drives"].replace(0, np.nan)
df["km_per_driving_day"] = df["driven_km_drives"] / df["driving_days"].replace(0, np.nan)
df["drives_per_driving_day"] = df["drives"] / df["driving_days"].replace(0, np.nan)
df.groupby("label")[["km_per_drive", "km_per_driving_day", "drives_per_driving_day"]].agg(["count", "median"])

km_per_drive             ... drives_per_driving_day          
                count     median  ...                  count    median
label                             ...                                 
churned          2522  73.491807  ...                   2143  7.454545
retained        11675  74.051037  ...                  11173  3.750000

[2 rows x 6 columns]

Median km/drive are 73.49 churned and 74.05 retained. Valid median km/driving-day are 523.09 and 272.63; drives/driving-day are 7.45 and 3.75. These are medians of user-level ratios, not ratios of group medians.

Zero denominators become NaN. Per-day ratios cover 2,143/2,536 churned users and 11,173/11,763 retained users. Exclusions are 15.50% versus 5.02%; differential coverage warrants sensitivity analysis. Investigate 1,024 rows with zero driving days but positive kilometers. High mileage does not identify occupation.

In [11]:
pd.crosstab(df.device, df.label, normalize="index")*100

label,churned,retained
device,,
Android,17.560110,82.439890
iPhone,17.831978,82.168022


Device-specific churn rates are 17.56% Android and 17.83% iPhone among known labels. The difference is 0.27 percentage points. No statistical-significance claim is made.

## Construct
Not applicable at this milestone.

## Execute
Investigate missing labels, inconsistent driving records and distribution tails. Confirm units, sampling and the operational churn definition. Establish observation and prediction windows to avoid temporal leakage. Future models should use a majority-class baseline and metrics that assess churn detection. Additional longitudinal activity and voluntary feedback may help explain patterns. Retention impact requires separate evaluation.

**Validation:** code executed sequentially in a fresh Python namespace with outputs regenerated. This was not a Jupyter-kernel validation.